In [3]:
import pandas as pd

In [4]:
clean_data_folder = "../data/cleaned/"
clean_data_path = clean_data_folder + "cleaned_data.csv"

df = pd.read_csv(clean_data_path)

In [5]:
df.shape

(180519, 60)

In [7]:
# BASELINE KPIs

# total orders
print(f"Total Orders: {df['Order Id'].nunique()}")
# total revenue
print(f"Total Revenue: {df['Sales'].sum()}")
# total profit
print(f"Total Profit: {df['Order Profit Per Order'].sum()}")
# late delivery rate
print(f"Late Delivery Rate: {df['late_delivery_flag'].mean()}")
# avg shipping variance
print(f"Avg Shipping Variance: {df['shipping_variance'].mean()}")
# avg order value
print(f"Avg Order Value: {df['Sales'].mean()}")

Total Orders: 65752
Total Revenue: 36784735.01337984
Total Profit: 3966902.974050357
Late Delivery Rate: 0.5727928916069777
Avg Shipping Variance: 0.5658074773292562
Avg Order Value: 203.77209608617287


In [31]:
shipping_analysis = (
    df.groupby("Shipping Mode")
      .agg(
          orders=("Order Id", "count"),
          delay_rate=("late_delivery_flag", "mean"),
          revenue=("Sales", "sum"),
          profit=("Order Profit Per Order", "sum")
      )
)
print(shipping_analysis)

                orders  delay_rate       revenue        profit
Shipping Mode                                                 
First Class      27814    1.000000  5.674370e+06  6.431219e+05
Same Day          9737    0.478279  1.942529e+06  2.030184e+05
Second Class     35216    0.797308  7.145445e+06  7.503082e+05
Standard Class  107752    0.397682  2.202239e+07  2.370454e+06


In [32]:
region_analysis = (
    df.groupby("Order Region")
      .agg(
          orders=("Order Id","count"),
          delay_rate=("late_delivery_flag","mean"),
          revenue=("Sales","sum")
      )
)
# sort regions by latest avg delivery time
print(region_analysis.sort_values(by=['delay_rate'], ascending=False))

                 orders  delay_rate       revenue
Order Region                                     
Central Africa     1677    0.607036  3.272630e+05
Western Europe    27109    0.585156  5.894381e+06
South Asia         7731    0.585047  1.553681e+06
South of  USA      4045    0.580964  7.857839e+05
Southeast Asia     9539    0.579830  1.932496e+06
East of USA        6915    0.579754  1.371112e+06
West Asia          6009    0.574971  1.174672e+06
East Africa        1852    0.574514  3.762349e+05
Eastern Europe     3920    0.574490  7.742666e+05
Central America   28341    0.572457  5.665712e+06
South America     14935    0.572347  2.960881e+06
Central Asia        553    0.571429  1.098399e+05
US Center          5887    0.571259  1.151356e+06
Eastern Asia       7280    0.567308  1.486401e+06
Southern Europe    9431    0.567278  2.047919e+06
North Africa       3232    0.566832  6.347522e+05
West of USA        7993    0.565995  1.571416e+06
Northern Europe    9792    0.564134  2.155831e+06


In [33]:
category_analysis = (
    df.groupby("Category Name")
      .agg(
          orders=("Order Id","count"),
          delay_rate=("late_delivery_flag","mean"),
          profit=("Order Profit Per Order","sum")
      )
)
# sort categories by latest avg delivery time
print(category_analysis.sort_values(by=['delay_rate'], ascending=False))

                      orders  delay_rate         profit
Category Name                                          
Golf Bags & Carts         61    0.688525    1810.069992
Lacrosse                 343    0.620991    4364.289984
Cameras                  592    0.619932   30289.799946
Pet Supplies             492    0.613821    3589.259959
Sporting Goods           357    0.599440   12518.610119
Basketball                67    0.597015    1845.670003
Fitness Accessories      309    0.595469    5258.390016
Crafts                   484    0.595041   25531.170060
Strength Training        111    0.594595     332.310091
Music                    434    0.594470   14436.319923
Boxing & MMA             423    0.593381    8641.370006
Consumer Electronics     431    0.591647   13223.399926
Accessories             1780    0.591573   16643.520074
As Seen on  TV!           68    0.588235     714.429980
Girls' Apparel          1201    0.586178   17288.569973
Women's Clothing         650    0.586154   19102

In [38]:
monthly_trends = (
    df.groupby("order_month")
      .agg(
          orders=("Order Id","count"),
          delay_rate=("late_delivery_flag","mean"),
          profit=("Order Profit Per Order","sum")
      ) 
)
# sort months by latest avg delivery time
print(monthly_trends.sort_values(by=['orders'], ascending=False))

             orders  delay_rate         profit
order_month                                   
January       17979    0.568663  367127.430615
May           15976    0.576302  337878.660269
July          15922    0.570280  348592.480100
March         15919    0.575853  333727.100652
August        15912    0.578997  360210.470630
September     15489    0.576603  359315.040248
April         15435    0.567153  339021.360056
June          15139    0.570579  324742.520726
February      14529    0.572579  301061.220935
October       12955    0.568506  331987.009971
December      12764    0.579442  281481.869563
November      12500    0.568480  281757.810287


In [40]:
(df['late_delivery_flag'] == df['Late_delivery_risk']).value_counts()

True     176096
False      4423
Name: count, dtype: int64

In [41]:
pd.crosstab(
    df['late_delivery_flag'],
    df['Late_delivery_risk']
)

Late_delivery_risk,0,1
late_delivery_flag,,
0,77119,0
1,4423,98977


In [56]:
# look further into cases where a delivery was not marked as a late delivery risk, but was delivered late anyway
mismatches = df[
    (df["late_delivery_flag"] == 1) &
    (df["Late_delivery_risk"] == 0)
]

mismatches[
    [
        "Delivery Status",
        "Order Status",
        "Days for shipping (real)",
        "Days for shipment (scheduled)",
        "shipping_variance",
        "Shipping Mode"
    ]
].head(20)

,Delivery Status,Order Status,Days for shipping (real),Days for shipment (scheduled),shipping_variance,Shipping Mode
5,Shipping canceled,CANCELED,6,4,2,Standard Class
10,Shipping canceled,SUSPECTED_FRAUD,6,2,4,Second Class
23,Shipping canceled,CANCELED,3,2,1,Second Class
39,Shipping canceled,CANCELED,1,0,1,Same Day
183,Shipping canceled,SUSPECTED_FRAUD,5,4,1,Standard Class
184,Shipping canceled,SUSPECTED_FRAUD,5,4,1,Standard Class
185,Shipping canceled,SUSPECTED_FRAUD,6,4,2,Standard Class
186,Shipping canceled,SUSPECTED_FRAUD,5,4,1,Standard Class
192,Shipping canceled,SUSPECTED_FRAUD,6,4,2,Standard Class
193,Shipping canceled,SUSPECTED_FRAUD,5,4,1,Standard Class


In [64]:
mismatch_analysis = (
    mismatches.groupby("Order Status")
      .agg(
          status=("Delivery Status", "count"),
          shipping_mode=("Shipping Mode", "count")
      )
)
print(mismatch_analysis)

                 status  shipping_mode
Order Status                          
CANCELED           2107           2107
SUSPECTED_FRAUD    2316           2316


In [65]:
pd.crosstab(
    mismatches['Order Status'],
    mismatches['Shipping Mode']
)

Shipping Mode,First Class,Same Day,Second Class,Standard Class
Order Status,,,,
CANCELED,615,101,497,894
SUSPECTED_FRAUD,686,102,594,934


In [66]:
mismatches[mismatches["Order Status"] == "CANCELED"]

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Product Status,shipping date (DateOrders),Shipping Mode,shipping_variance,late_delivery_flag,early_delivery_flag,ontime_delivery_flag,order_year,order_month,order_quarter
5,TRANSFER,6,4,18.580000,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,0,2018-01-19 11:03:00,Standard Class,2,1,0,0,2018,January,1
23,TRANSFER,3,2,17.700001,294.980011,Shipping canceled,0,73,Sporting Goods,Caguas,...,0,2018-01-16 04:45:00,Second Class,1,1,0,0,2018,January,1
39,TRANSFER,1,0,57.299999,304.809998,Shipping canceled,0,73,Sporting Goods,Newark,...,0,2018-01-13 11:09:00,Same Day,1,1,0,0,2018,January,1
200,TRANSFER,6,4,74.040001,203.970001,Shipping canceled,0,17,Cleats,Vista,...,0,2016-09-01 12:38:00,Standard Class,2,1,0,0,2016,August,3
203,TRANSFER,5,4,85.050003,189.000000,Shipping canceled,0,24,Women's Apparel,Fond Du Lac,...,0,2016-10-03 10:19:00,Standard Class,1,1,0,0,2016,September,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180109,TRANSFER,2,1,176.389999,359.980011,Shipping canceled,0,45,Fishing,Fort Lauderdale,...,0,2016-02-10 11:31:00,First Class,1,1,0,0,2016,February,1
180279,TRANSFER,6,4,76.800003,299.989990,Shipping canceled,0,45,Fishing,Dallas,...,0,2016-02-03 21:10:00,Standard Class,2,1,0,0,2016,January,1
180322,TRANSFER,2,1,-289.910004,377.980011,Shipping canceled,0,45,Fishing,Caguas,...,0,2016-01-28 08:12:00,First Class,1,1,0,0,2016,January,1
180323,TRANSFER,2,1,-272.660004,371.980011,Shipping canceled,0,45,Fishing,Caguas,...,0,2016-01-28 08:12:00,First Class,1,1,0,0,2016,January,1


In [67]:
mismatches['Order Status'].value_counts()

Order Status
SUSPECTED_FRAUD    2316
CANCELED           2107
Name: count, dtype: int64

In [79]:
category_summary = (
    df.groupby("Category Name")
        .agg(
            total_orders=("Order Id", "count"),
            suspected_fraud_orders=(
                "Order Status",
                lambda x: (x == "SUSPECTED_FRAUD").sum()
            )
        )
)
category_summary["suspected_fraud_pct"] = (
    category_summary["suspected_fraud_orders"]
    / category_summary["total_orders"]
    * 100
)
category_summary = category_summary.sort_values(
    by="suspected_fraud_pct",
    ascending=False
)
category_summary

,total_orders,suspected_fraud_orders,suspected_fraud_pct
Category Name,,,
Basketball,67,3,4.477612
Women's Golf Clubs,181,8,4.419890
Kids' Golf Clubs,384,13,3.385417
Sporting Goods,357,12,3.361345
DVDs,483,16,3.312629
Health and Beauty,362,11,3.038674
Girls' Apparel,1201,35,2.914238
Baby,207,6,2.898551
Boxing & MMA,423,12,2.836879


In [8]:
late_orders = (
    df[df["Late_delivery_flag"] == 1]["Order Id"]
    .nunique()
)

late_orders

KeyError: 'Late_delivery_flag'